<a href="https://colab.research.google.com/github/iliaxant/Pattern_Recognition_Final_Project/blob/main/Final_Project_PR_58545.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Τελική Εργασία Εξαμήνου**

## Αναγνώριση Προτύπων - Ακαδημαϊκό έτος 2025-2026

## Ηλίας Ξανθόπουλος 58545

## GitHub Repo: https://github.com/iliaxant/Pattern_Recognition_Final_Project

---

## **Set-Up**

1) Χειροκίνητο ανέβασμα του αρχείου *Final_Project_data.zip*.

2) Unziping του αρχείου *Final_Project_data.zip*:

In [1]:
import zipfile

zip_path = '/content/Final_Project_data.zip'
zip_ref = zipfile.ZipFile(zip_path, 'r')
zip_ref.extractall('/content')
zip_ref.close()

print("Data unzipped successfully to /content directory.")

Data unzipped successfully to /content directory.


3) Εγκατάσταση και φόρτωση των απαραίτητων βιβλιοθηκών.

In [83]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random
import re
from scipy.stats import randint

from sklearn.model_selection import (StratifiedKFold, GridSearchCV, cross_val_predict,
                                     RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             precision_score, f1_score, average_precision_score)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

4) Ορισμός συνάρτησης αρχικοποίησης όλων των χρησιμοποιούμενων random state machines για reproducibility.

In [57]:
def set_seed(seed=15):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

---

## **Μέρος 1ο – Ανίχνευση Αστοχίας σε Βιομηχανική Διαδικασία**

Αρχικοποίηση των random state machines με το seed '15'.

In [58]:
state = 15
set_seed(state)

print(f"Seed '{state}' has been set.")

Seed '15' has been set.


### **Α) Επισκόπηση Δεδομένων**


Ορισμός συνάρτησης ανάλυσης δεδομένων ενός dataframe.

Η συνάρτηση **dataframe_analysis()** δέχεται ως παραμέτρους εισόδου:

*   **df:** το pandas dataframe προς ανάλυση.
*   **label_flag=False:** σημαία που δηλώνει αν υπάρχει στήλη ετικετών στο dataframe.
*   **label_col_name=' '** string που αντιστοιχεί στο όνομα της στήλης ετικετών,εφόσον υπάρχει.

Οι πληροφορίες που δίνει η συνάρτηση για τα δεδομένα αφορούν:

*   Αναλογία των κλάσεων (εφόσον label_flag=Τrue)
*   Missing Values
*   Διπλότυπα δείγματα
*   Features με σταθερή τιμή
*   Features με σχεδόν σταθερή τιμή (πλήθος σταθερών τιμών ίσο ή μεγαλύτερο του 99% των δειγμάτων)

In [59]:
def dataframe_analysis(df, label_flag=False, label_col_name=''):

    print(df.info())

    if label_flag:
        print('\n' + 15 * "-" + " Class Info " + 15 * "-" + '\n')
        print(df[label_col_name].value_counts())
        print()
        print(df[label_col_name].value_counts(normalize=True).map('{:.1%}'.format))
        print('\n' + 42 * "-")


    print('\n' + 15 * "-" + " Missing Values " + 15 * "-" + '\n')

    missing_per_feat = df.isnull().sum()
    missing_per_sample = df.isnull().sum(axis=1)

    missing_per_feat_percent = (df.isnull().mean() * 100).round(2)
    missing_per_sample_percent = (df.isnull().mean(axis=1) * 100).round(2)

    missing_feat_table = pd.DataFrame({
        'Missing Count': missing_per_feat,
        'Percent (%)': missing_per_feat_percent
    })

    missing_sample_table = pd.DataFrame({
        'Missing Count': missing_per_sample,
        'Percent (%)': missing_per_sample_percent
    })

    if label_flag and label_col_name in df.columns:
        missing_sample_table['Class'] = df[label_col_name]

    missing_feat_table = missing_feat_table.sort_values(by='Percent (%)', ascending=False)
    missing_sample_table = missing_sample_table.sort_values(by='Percent (%)', ascending=False)


    print(f'Missing Values per Feature Table (Descending):')
    print(missing_feat_table)
    print()

    print(f'Τop 20 Missing Values per Feat Table:')
    print(missing_feat_table.head(20))
    print()

    print(f'Missing Values per Sample Table (Descending):')
    print(missing_sample_table)
    print()

    print(f'Τop 20 Missing Values per Sample Table:')
    print(missing_sample_table.head(20))
    print()

    if label_flag:

        classes = df[label_col_name].unique()
        for cls in sorted(classes):

            class_subset = df[df[label_col_name] == cls]
            total_missing_class = class_subset.isnull().sum().sum()
            total_cells_class = class_subset.size
            percent_missing_class = (total_missing_class / total_cells_class) * 100

            print(f"Missing values of Class {cls}: {total_missing_class} / {total_cells_class} "
                  f"-> {percent_missing_class:.2f}%")


    total_cells = df.size
    total_missing = missing_per_feat.sum()
    percent_missing = (total_missing / total_cells) * 100

    print(f"\nMissing values of set: {total_missing} / {total_cells} "
          f"-> {percent_missing:.2f}%")

    print('\n' + 46 * "-")


    print('\n' + 15 * "-" + " Duplicates " + 15 * "-" + '\n')
    dupes = df.duplicated()
    print(f'Dublicates List:')
    print(dupes)

    print(f"\nTotal dublicate samples: {dupes.sum()}")
    print('\n' + 42 * "-")


    print('\n' + 15 * "-" + " Constant Features " + 15 * "-" + '\n')
    constant_feat = [feat for feat in df.columns if df[feat].nunique() <= 1]
    print(f"Constant Features List: {constant_feat}")
    print(f'\nTotal constant features: {len(constant_feat)}')
    print('\n' + 48 * "-")


    print('\n' + 15 * "-" + " Quasi-Constant Features " + 15 * "-" + '\n')

    quasi_constant_feats = []
    for feat in df.columns:

        if (feat == label_col_name) or (feat in constant_feat): continue

        top_val_freq = df[feat].value_counts(normalize=True, dropna=False).values[0]

        if top_val_freq >= 0.99:
            quasi_constant_feats.append(feat)

    print(f"Quasi-Constant (>=99% but not 100% of values) Feature List: {quasi_constant_feats}")
    print(f'\nTotal quasi-constant features: {len(quasi_constant_feats)}')

    print('\n' + 55 * "-")



Φόρτωση των training δεδομένων (*Training_data_manifacturing.csv* αρχείο). Ονομασία της κάθε στήλης (καθαρά προαιρετικό) και εκτύπωση χρήσιμων πληροφοριών για τα training δεδομένα μέσω της **dataframe_analysis()**.

In [60]:
columns = [f"Feat {i}" for i in range(474)] + ["y"]

class_column = columns[-1]
data_columns = columns[0:474]

df_train = pd.read_csv("Training_data_manifacturing.csv", header=None, names=columns)

print('\n' + 25 * "=" + " Training Data " + 25 * "=" + '\n')
dataframe_analysis(df_train, 1, class_column)
print('\n' + 65 * "=" + '\n\n')



========================= Training Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Columns: 475 entries, Feat 0 to y
dtypes: float64(474), int64(1)
memory usage: 4.5 MB
None

--------------- Class Info ---------------

y
0    1170
1      84
Name: count, dtype: int64

y
0    93.3%
1     6.7%
Name: proportion, dtype: object

------------------------------------------

--------------- Missing Values ---------------

Missing Values per Feature Table (Descending):
          Missing Count  Percent (%)
Feat 148           1153        91.95
Feat 247           1153        91.95
Feat 149           1153        91.95
Feat 248           1153        91.95
Feat 401           1077        85.89
...                 ...          ...
Feat 418              0         0.00
Feat 459              0         0.00
Feat 417              0         0.00
Feat 18               0         0.00
y                     0         0.00

[475 rows x 2 columns]

Τop 20 M

Φόρτωση των testing δεδομένων (*Test_data_manifacturing.csv* αρχείο) και εκτύπωση χρήσιμων πληροφοριών για τα δεδομένα μέσω της **dataframe_analysis()**.

**ΣΧΟΛΙΟ:** Αν και εφαρμόζεται η συνάρτηση επισκόπησης και στα testing δεδομένα, παρακάτω δεν πρέπει γίνει καμία επεξεργασία σε οποιοδήποτε από τα set με βάση τις πληροφορίες που δίνει η συνάρτηση για τα testing δεδομένα. Ό,τι επεξεργασία ακολουθήσει πάνω στα training και testing δεδομένα πρέπει να γίνει με γνώμονα **ΜΟΝΟ** το training set. Η χρήση της συνάρτησης επισκόπησης στα testing δεδομένα γίνεται καθαρά για την κατανόηση του προβλήματος προς επίλυση.

In [61]:
df_test = pd.read_csv("Test_data_manifacturing.csv", header=None, names=data_columns)

print('\n' + 25 * "=" + " Testing Data " + 25 * "=" + '\n')
dataframe_analysis(df_test)
print('\n' + 64 * "=" + '\n\n')



========================= Testing Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Columns: 474 entries, Feat 0 to Feat 473
dtypes: float64(473), int64(1)
memory usage: 1.1 MB
None

--------------- Missing Values ---------------

Missing Values per Feature Table (Descending):
          Missing Count  Percent (%)
Feat 248            276        88.18
Feat 247            276        88.18
Feat 149            276        88.18
Feat 148            276        88.18
Feat 401            264        84.35
...                 ...          ...
Feat 469              0         0.00
Feat 470              0         0.00
Feat 471              0         0.00
Feat 472              0         0.00
Feat 473              0         0.00

[474 rows x 2 columns]

Τop 20 Missing Values per Feat Table:
          Missing Count  Percent (%)
Feat 248            276        88.18
Feat 247            276        88.18
Feat 149            276        88.18
Feat 148    

Παρατηρούνται δύο πολύ σημαντικά προβλήματα στα δεδομένα.

Το πρώτο είναι τα missing values. Τόσο στα training όσο και στα testing δεδομένα υπάρχει ένα ποσοστό ~5.54% των κελιών το οποίο δεν έχει καθόλου τιμή. Αν και φαίνεται αρκετά μικρό το ποσοστό, οι missing τιμές είναι συγκεντρωμένες σε υψηλά ποσοστά σε συγκεκριμένα χαρακτηριστικά, με κάποια από αυτά να φτάνουν missing ποσοστά εώς και 91.95% στο training και 88.18% στο testing. Παρόλα αυτά, όπως φαίνεται και στις πληροφορίες για τα training δεδομένα, δεν υπάρχουν καθόλου NaN κελία στην στήλη των ετικετών και δεν υπάρχουν δείγματα με υπερβολικά πολλές missing τιμές, το οποίο σημαίνει ότι δεν απαραίτητο να απορριφθούν δείγματα, αλλά πρέπει οπωσδήποτε να επιλεχθεί μια στρατηγική αντιμετώπισης των missing values.

Το δεύτερο πρόβλημα αφορά την κατανομή των κλάσεων και είναι η ανισότητα. Όπως φαίνεται παραπάνω, η συντριπτική πλειοψηφία των δειγμάτων του training dataframe ανήκει στην κλάση 0 και μόνο το 6.7% των training δεδομένων αντιστοιχεί στην κλάση 1. Παρόλο που δεν υπάρχουν ετικέτες για επιβεβαίωση, είναι εκ των προτέρων γνωστό ότι παρόμοια αναλογία υπάρχει και στα testing δεδομένα. Επομένως, πρέπει να εφαρμοστούν οι κατάλληλες τεχνικές για την διαχείριση της ανισότητας κλάσεων.

Και τα δύο προβλήματα επιλύονται σε παρακάτω cells.

Επίσης σημαντική παρατήρηση είναι η ύπαρξη σχεδόν σταθερών (και σταθερών) χαρακτηριστικών στo training set. Αυτή η σταθερότητα καθιστά τα χαρακτηριστικά redundant και οδηγούν σε πιο περίπλοκους τους υπολογισμούς χωρίς κάποια πρακτική αξία. Επομένως, αυτά τα χαρακτηριστικά είναι αναγκαίο να αφαιρεθούν.

**ΣΧΟΛΙΟ:** Στα παραπάνω μηνύματα για τα δεδομένα υπάρχει ένα discrepancy ως προς τον τύπο των δεδομένων.

Παρατηρούμε ότι στα training δεδομένα όλες οι στήλες είναι float64 εκτός από την στήλη των ετικετών που είναι int64. Στα testing δεδομένα που υπάρχουν τα ίδια χαρακτηριστικά αλλά δεν υπάρχει η στήλη ετικετών, πάλι βλέπουμε όλες τις στήλες να είναι float64, εκτός από μία που είναι int64.

Αυτό εν τέλει δεν είναι τυχαίο, διότι η ύπαρξη missing τιμών σε μία στήλη αυτόματα την μετατρέπει σε τύπο δεδομένων float64. Επομένως, στην περίπτωση των training δεδομένων, όλες οι στήλες χαρακτηριστικών τύπου int64 έχουν NaN τιμές, οπότε μετατρέπεται η στήλη σε float64, ενώ στα testing δεδομένα υπάρχει μια στήλη int64 που προηγουμένως είχε missing τιμές, αλλά τώρα δεν έχει, οπότε παραμένει int64.

Αυτό επιβεβαιώνεται τόσο από το παρακάτω code cell, όσο και από μια γρήγορη επισκόπηση των τιμών των δύο *.csv* αρχείων.




In [62]:
print("*** Ignore 'dtype' lines ***\n")

int_cols_test = df_test.select_dtypes(include=['int64']).columns
print(f"Columns of int64 data type: {(list(int_cols_test))}")

print(f"\n--- Info for '{int_cols_test}' on Train data frame ---")
print(f"Data type: {df_train[int_cols_test].dtypes}")
print(f"Num of missing values: {df_train[int_cols_test].isnull().sum()}")

print(f"\n--- Info for '{int_cols_test}' on Test data frame ---")
print(f"Data type: {df_test[int_cols_test].dtypes}")
print(f"Num of missing values: {df_test[int_cols_test].isnull().sum()}")

*** Ignore 'dtype' lines ***

Columns of int64 data type: ['Feat 128']

--- Info for 'Index(['Feat 128'], dtype='object')' on Train data frame ---
Data type: Feat 128    float64
dtype: object
Num of missing values: Feat 128    5
dtype: int64

--- Info for 'Index(['Feat 128'], dtype='object')' on Test data frame ---
Data type: Feat 128    int64
dtype: object
Num of missing values: Feat 128    0
dtype: int64


### **Β) Προεπεξεργασία Δεδομένων**

Μετά από την επισκόπηση των δεδομένων ακολουθεί η προεπεξεργασία τους. Η διαδικασίες αυτού τα σταδίου επιλέγονται με βάση της πληροφορίες που έχουν αντληθεί από την επισκόπηση και είναι oi εξής:

*   **Αφαίρεση Σταθερών και Σχεδόν Σταθερών Χαρακτηριστικών**
*   **Αφαίρεση Χαρακτηριστικών με Μεγάλο Ποσοστό Ελλειπουσών Τιμών**
*   **Διαχείριση Ελλειπουσών Τιμών**
*   **Standard Scaling**
*   **Principal Component Analysis (PCA)**

Οι παραπάνω διαδικασίες εφαρμόζονται με την σειρά που παρατίθενται, αλλά απευθείας εκτελούνται μόνο οι δύο πρώτες. Οι υπόλοιπες τρεις εφαρμόζονται πριν την εκπαίδευση του μοντέλου σε κάθε fold και πριν την τελική επανεκπαίδευση. Για αυτό και σε αυτή την ενότητα, για τις τρεις διεργασίες ορίζονται μόνο οι συναρτήσεις που τις διαχειρίζονται, αφού αργότερα θα προστεθούν αυτές στο pipeline της εκπαίδευσης. Το γιατί επιλέγεται να αξιοποιηθεί k-fold cross validation και πως δομείται το pipeline εκπαίδευσης εξηγείται σε επόμενη f;ash. Σε αυτή την ενότητα ορίζονται μόνο οι διαδικασίες preprocessing που εφαρμόζονται.



#### **i) Αφαίρεση Σταθερών και Σχεδόν Σταθερών Χαρακτηριστικών**




Αφαίρεση των σταθερών και των σχεδόν σταθερών (>=99% σταθερών) χαρακτηριστικών του training set, τόσο από τα training, όσο και τα testing δεδομένα. Αυτό το βήμα πρέπει να γίνει, καθώς τα σταθερά χαρακτηριστικά δεν προσφέρουν καμία πληροφορία στο μοντέλο αλλά αυξάνοντας μόνο τις διαστάσεις του προβλήματος.

In [63]:
quasi_constant_thresh = 0.99

quasi_constant_feats = []
for feat in df_train[data_columns].columns:

    top_val_freq = df_train[feat].value_counts(normalize=True, dropna=False).values[0]

    if top_val_freq >= quasi_constant_thresh:
        quasi_constant_feats.append(feat)

print(f"\nConstant / Quasi-Constant (>= {quasi_constant_thresh*100}% same value) "
      f"Feature list:")
print(quasi_constant_feats)
print(f"\nRemoving {len(quasi_constant_feats)} Constant / Quasi-Constant Features...")

df_train_const_clean = df_train.drop(columns=quasi_constant_feats)
df_test_const_clean = df_test.drop(columns=quasi_constant_feats)

print(f'\nOriginal Train shape: {df_train.shape}')
print(f'Original Test shape: {df_test.shape}')

print(f'\nTrain shape after feature removal: {df_train_const_clean.shape}')
print(f'Test shape after feature removal: {df_test_const_clean.shape}\n')



Constant / Quasi-Constant (>= 99.0% same value) Feature list:
['Feat 68', 'Feat 188', 'Feat 191', 'Feat 287', 'Feat 292', 'Feat 388']

Removing 6 Constant / Quasi-Constant Features...

Original Train shape: (1254, 475)
Original Test shape: (313, 474)

Train shape after feature removal: (1254, 469)
Test shape after feature removal: (313, 468)



#### **ii) Αφαίρεση Χαρακτηριστικών με Μεγάλο Ποσοστό Ελλειπουσών Τιμών**

Χαρακτηριστικά του training set με μεγάλο ποσοστό missing τιμών (>=80%) πρέπει να αφαιρεθούν από το training και testing set γιατί δεν είναι εφικτό να συμπληρωθούν τα κενά χωρίς βλάβη της γενικότητας και η μείωση των διαστάσεων προσφέρει στην συγκεκριμένη περίπτωση μεγαλύτερη αξία. Προτού όμως απομακρυνθούν αυτές οι στήλες είναι απαραίτητο να προσδιοριστεί εάν αυτές οι missing τιμές συσχετίζονται περισσότερο με μία από τις δύο κλάσεις, που σημαίνει ότι αυτό το υψηλό ποσοστό missing προσφέρει χρήσιμες για το μοντέλο πληροφορίες. Αυτή η κατηγορία missing data ονομάζεται **MNAR** και αναλύεται παρακάτω στην υποενότητα ***Διαχείριση Ελλειπουσών Τιμών***. Εάν το feature δεν ανήκει σε αυτή την κατηγορία, τότε μπορεί να αφαιρεθεί από τα δεδομένα.

Εύρεση των χαρακτηριστικών με missing data >=80% στο training set. Έλεγχος κάθε εντοπισμένου χαρακτηριστικού για το αν τα missing δεδομένα του ανήκουν στην κατηγορία NMAR συγκρίνοντας τα ποσοστά missing data μεταξύ των κλάσεων. Αν η διαφορά μεταξύ των δύο ποσοστών είναι <20%, τότε το χαρακτηριστικό αφαιρείται από τα training και testing δεδομένα, διαφορετικά διατηρείται.

In [64]:
missing_thresh = 0.80
missing_diff_thresh = 0.10

limit = missing_thresh * len(df_train_const_clean)
high_missing_feats = df_train_const_clean.columns[df_train_const_clean.isnull().sum() > limit].tolist()

print(f"\nMissing >={missing_thresh*100}% of values Feature list:")
print(high_missing_feats)


print('\n' + 10 * "-" + " MNAR Check " + 10 * "-" + '\n')

idx_0 = df_train_const_clean['y'] == 0
idx_1 = df_train_const_clean['y'] == 1

suspicious_feats = []

for feat in high_missing_feats:

    missing_rate_0 = df_train_const_clean.loc[idx_0, feat].isnull().mean()
    missing_rate_1 = df_train_const_clean.loc[idx_1, feat].isnull().mean()

    diff = missing_rate_1 - missing_rate_0
    print(f"{feat:<8}: Missing Rate Class 0(-) = {missing_rate_0:>6.2%} | Class 1(+) = {missing_rate_1:>6.2%} "
          f"| Diff = {diff:>7.2%}")

    if diff > missing_diff_thresh:
        suspicious_feats.append(feat)

print(f"\nSummary: Possible MNAR (Diff >= {missing_diff_thresh*100}%) -> ")
print(suspicious_feats)

print('\n' + 32 * "-")


feats_to_drop = [feat for feat in high_missing_feats if feat not in suspicious_feats]

print(f"\nRemoving {len(feats_to_drop)} features:")
print(feats_to_drop)

df_train_missing_clean = df_train_const_clean.drop(columns=feats_to_drop)
df_test_missing_clean = df_test_const_clean.drop(columns=feats_to_drop)

print(f'\nCurrent Train shape: {df_train_const_clean.shape}')
print(f'Current Test shape: {df_test_const_clean.shape}')

print(f'\nTrain shape after feature removal: {df_train_missing_clean.shape}')
print(f'Test shape after feature removal: {df_test_missing_clean.shape}\n')



Missing >=80.0% of values Feature list:
['Feat 79', 'Feat 148', 'Feat 149', 'Feat 202', 'Feat 247', 'Feat 248', 'Feat 303', 'Feat 401']

---------- MNAR Check ----------

Feat 79 : Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 148: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 149: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 202: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 247: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 248: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 303: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 401: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%

Summary: Possible MNAR (Diff >= 10.0%) -> 
[]

--------------------------------

Removing 8 features:
['Feat 79', 'Feat 148', 'Feat 149', 'Feat 202', 'Feat 247', 'Feat 248

Από τα παραπάνω αποτελέσματα γίνεται ξεκάθαρο ότι τουλάχιστον από τα χαρακτηριστικά με μεγαλύτερο ποσοστό από 80% missing data, κανένα δεν διαθέτει MNAR ελλειπείς τιμές, οπότε αυτά μπορούν να αφαιρεθούν άφοβα.


#### **iii) Διαχείριση Ελλειπουσών Τιμών**

Προτού εφαρμοστεί η κατάλληλη τεχνική διαχείρισης των missing values, είναι αναγκαίο να προσδιοριστεί η φύση των δεδομένων που λείπουν. Υπάρχουν λοιπόν τρεις κατηγορίες:

*   **Missing Completely At Random (MCAR):** Σε αυτήν την περίπτωση, η πιθανότητα έλλειψης των δεδομένων είναι ανεξάρτητη από τις αντίστοιχες τιμές (παρατηρούμενες ή μη) των χαρακτηριστικών. Για παράδειγμα, στην παρούσα εφαρμογή θα μπορούσε να ήταν missing τιμές λόγω στιγμιαίων διακοπών ρεύματος στους αισθητήρες, κάτι που οφείλεται σε έναν εξωγενή και τελείως τυχαίο παράγοντα.

*   **Missing At Random (MAR):** Εδώ οι missing τιμές χαρακτηριστικών οφείλονται στις μετρούμενες τιμές άλλων χαρακτηριστικών. Κάτι τέτοιο ισχύει όταν, για παράδειγμα, ένας αισθήτηρας σταματάει να καταγράφει τιμές (άρα missing value), όταν ένας άλλος αισθητήρας καταγράφει πάρα πολύ ψηλή τιμή ενός άλλου μεγέθους. Σε αυτήν την περίπτωση, τα ελλειπή δεδομένα δεν οφείλονται στην ίδια την τιμή που κανονικά θα παρατηρούταν, αλλά σε άλλα χαρακτηριστικά για τα οποία έχει καταγραφεί τιμή.

*   **Missing Not At Random (MNAR):** Η έλλειψη τιμής οφείλεται στην ίδια τιμή που λείπει. Εδώ η αιτία δεν είναι καθόλου τυχαία, αλλά εξαρτάται από την ίδια την φύση του μετρούμενου μεγέθους. Στην συγκεκριμένη εφαρμογή, θα μπορούσε να υπάρχει αυτός ο τύπος, εάν ένας αισθητήρας παύει να λειτουργεί, όταν η μετρούμενη τιμή ξεπερνάει ένα όριο.

Από αυτές τις τρείς κατηγορίες, η πιο διαχειρίσιμη είναι η πρώτη, καθώς είναι εντελώς τυχαία, ενώ πιο επικίνδυνη είναι η τρίτη, διότι μπορεί να φανερώνει πληροφορίες για τις τάξεις που προσπαθούμε να διαχωρίσουμε. Κάθε τύπος απαιτεί διαφορετική τεχνική αντιμετώπισης, επομένως πρέπει να προσδιοριστεί ο τύπος των ελλειπουσών τιμών.

#### **iv) Standard Scaling**

Ορισμός συνάρτησης κανονικοποίησης δεδομένων ενός dataframe.

Η συνάρτηση **data_normalization()** δέχεται ως παραμέτρους εισόδου:

*   **X_train:** το pandas dataframe του training set.
*   **X_val:** το pandas dataframe του validation set.
*   **verbose=0** flag για το αν θα τυπωθούν μυνήματα για τον μέσο όρο και της std του X_train.

και επιστρέφει:

*   **X_train_n:** το pandas dataframe του κανονικοποιημένου training set.
*   **X_val_n:** το pandas dataframe του κανονικοποιημένου validation set.

Η συνάρτηση εκτελεί standard scaling στα training και testing δεδομένα κάνοντας χρήση του μέσου όρου και της std μόνο των δεδομένων training.

### **Γ) Διαχωρισμός Δεδομένων**




Τα δεδομένα είναι εξαρχής χωρισμένα σε training και testing sets. Ωστόσο, επείδη η κλάση 1 αποτελεί μόνο το 6.7% του training set, η εκπαίδευση με Holdout Validation καθίσταται μη βοηθητική. Προκειμένου να αξιοποιηθεί στην εκπαίδευση η κλάση 1 όσο πιο πολύ γίνεται, προτιμάται η εκπαίδευση με Stratified K-Fold Validation. K-Fold έτσι ώστε όλο το πλήθος των δεδομένων και κυρίως της κλάσης 1 να συμμετέχει στην εκπαίδευση του μοντέλου, και Stratified έτσι ώστε να μπορέσει να γίνει μια ομοιόμορφη εκπαίδευση σε καθένα από τα folds.

Επομένως, το testing set παραμένει το ίδιο, αλλά το training set χωρίζεται περαιτέρω σε 5 stratified folds.

Διαχωρισμός του training set σε 5 stratified folds για την εφαρμογή 5-Fold Cross Validation και εκτύπωση χρήσιμων πληροφοριών για κάθε fold.

In [65]:
n_folds = 5

X = df_train_missing_clean[df_train_missing_clean.columns[:-1]]
y = df_train_missing_clean[df_train_missing_clean.columns[-1]]

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=state)

print("\nTraining set before folds:", X.shape)
print('\n' + 15 * "=" + " Fold Info " + 15 * "=" + '\n')

fold_num = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    print(10 * "-" + f" Fold {fold_num} " + 10 * "-")
    print(f"Training set:   {X_train.shape}")
    print(f"Validation set: {X_val.shape}")

    train_c_count = np.bincount(y_train)
    val_c_count = np.bincount(y_val)

    print(f"Training class counts:   {np.bincount(y_train)}-> [{(100*train_c_count[0]/X_train.shape[0]):.2f}% "
          f"{(100*train_c_count[1]/X_train.shape[0]):.2f}%]")
    print(f"Validation class counts: {np.bincount(y_val)}-> [{(100*val_c_count[0]/X_val.shape[0]):.2f}% "
          f"{(100*val_c_count[1]/X_val.shape[0]):.2f}%]\n")

    fold_num += 1

print( 41 * "=" + '\n\n')



Training set before folds: (1254, 460)

=============== Fold Info ===============

---------- Fold 1 ----------
Training set:   (1003, 460)
Validation set: (251, 460)
Training class counts:   [936  67]-> [93.32% 6.68%]
Validation class counts: [234  17]-> [93.23% 6.77%]

---------- Fold 2 ----------
Training set:   (1003, 460)
Validation set: (251, 460)
Training class counts:   [936  67]-> [93.32% 6.68%]
Validation class counts: [234  17]-> [93.23% 6.77%]

---------- Fold 3 ----------
Training set:   (1003, 460)
Validation set: (251, 460)
Training class counts:   [936  67]-> [93.32% 6.68%]
Validation class counts: [234  17]-> [93.23% 6.77%]

---------- Fold 4 ----------
Training set:   (1003, 460)
Validation set: (251, 460)
Training class counts:   [936  67]-> [93.32% 6.68%]
Validation class counts: [234  17]-> [93.23% 6.77%]

---------- Fold 5 ----------
Training set:   (1004, 460)
Validation set: (250, 460)
Training class counts:   [936  68]-> [93.23% 6.77%]
Validation class counts:

### **Δ) Βελτιστοποίση Υπερπαραμέτρων**


#### **i) Pipeline Εκπαίδευσης**

In [81]:
search_pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(bootstrap=True, random_state=state,
                                   n_jobs=-1))
])

#### **ii) Αναζήτηση στον χώρο των Υπερπαραμέτρων**

In [87]:
param_distributions = {
    'imputer__strategy': ['median', 'most_frequent'],
    'clf__n_estimators': randint(200, 1200),
    'clf__max_depth': randint(10, 25),
    'clf__min_samples_leaf': randint(2, 15),
    'clf__min_samples_split': randint(2, 20),
    'clf__max_features': ['sqrt', 'log2', 0.1, 0.4, 0.7],
    'clf__class_weight': ['balanced', 'balanced_subsample']
}

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=state,
    verbose=1
)

random_search.fit(X, y)

best_model_search = random_search.best_estimator_
cv_results = pd.DataFrame(random_search.cv_results_)
sorted_results = cv_results.sort_values(by='rank_test_score')

print("\n" + 10 * "=" + " Random Search Model Ranking " +  10 * "="  + "\n")

for index, row in sorted_results.iterrows():
    print(f"Rank: {row['rank_test_score']}")
    print(f"Score (ROC AUC): {row['mean_test_score']:.4f}")
    print("Parameters:")

    for param, value in row['params'].items():
        print(f"   {param}: {value}")

    print("\n")

print( 49 * "=" + "\n")

print("Random Search Best Model:")
print(best_model_search)
print(f"\nRandom Search Best Params (ROC AUC = {random_search.best_score_:.4f}):")
print(random_search.best_params_)
print()


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best Params: {'clf__class_weight': 'balanced', 'clf__max_depth': 15, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 7, 'clf__min_samples_split': 2, 'clf__n_estimators': 356, 'imputer__strategy': 'most_frequent'}
Best ROC AUC: 0.749934012066365


In [86]:
n_classifiers = 200
rf_max_depth = 10
rf_min_samples_leaf = 2
rf_min_samples_split = 5
rf_max_features = 'sqrt'
bootstrapping = True

results = {
    'roc_auc': [],
    'pr_auc': [],
    'optimal_threshold': [],
    'precision': [],
    'recall': [],
    'f1': []
}

imputer = SimpleImputer(strategy='most_frequent')
scaler = StandardScaler()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)

    X_train_n = scaler.fit_transform(X_train_imp)
    X_val_n = scaler.transform(X_val_imp)

    X_train_final = X_train_n
    X_val_final = X_val_n
    y_train_final = y_train
    y_val_final = y_val

    classifier = RandomForestClassifier(
        n_estimators=n_classifiers,
        max_depth=rf_max_depth,
        min_samples_leaf=rf_min_samples_leaf,
        min_samples_split=rf_min_samples_split,
        max_features=rf_max_features,
        bootstrap=bootstrapping,
        class_weight='balanced_subsample',
        random_state=state,
        n_jobs=-1
    )

    classifier.fit(X_train_final, y_train_final)

    y_prob = classifier.predict_proba(X_val_final)[:, 1]

    roc_auc = roc_auc_score(y_val_final, y_prob)
    pr_auc = average_precision_score(y_val_final, y_prob)

    precisions, recalls, thresholds = precision_recall_curve(y_val_final, y_prob)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

    best_f1_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_f1_idx]
    best_f1 = f1_scores[best_f1_idx]

    y_pred = (y_prob >= best_threshold).astype(int)

    rec = 100 * recall_score(y_val_final, y_pred)
    prec = 100 * precision_score(y_val_final, y_pred, zero_division=0)

    results['roc_auc'].append(roc_auc)
    results['pr_auc'].append(pr_auc)
    results['optimal_threshold'].append(best_threshold)
    results['precision'].append(prec)
    results['recall'].append(rec)
    results['f1'].append(best_f1)

    cm = confusion_matrix(y_val_final, y_pred)

    print(f"\n--- Fold {fold+1} ---")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"PR AUC: {pr_auc:.4f}")
    print(f"Optimal (using F1-score) Threshold: {best_threshold:.4f}")
    print(f"F1 Score: {100 * best_f1:.2f}%")
    print(f"Precision: {prec:.2f}%  |  Recall: {rec:.2f}%")
    print("Confusion Matrix:\n", cm)


print("\n" + 15 * "=" + " Training Summary " + 15 * "=" + '\n')
print(f"Average ROC AUC (Random Classifier = 0.5000): {np.mean(results['roc_auc']):.4f}")
print(f"Average PR AUC (Baseline = {np.mean(y):.4f}): {np.mean(results['pr_auc']):.4f}")
print(f"Average Optimal Threshold: {np.mean(results['optimal_threshold']):.4f}\n")

print("--- After Optimal Threshold ---")
print(f"Average Precision: {np.mean(results['precision']):.2f}%")
print(f"Average Recall:    {np.mean(results['recall']):.2f}%")
print(f"Average F1 Score:  {np.mean(results['f1'])*100:.2f}%")
print("\n" + 48* "=" + "\n")



--- Fold 1 ---
ROC AUC: 0.7592
PR AUC: 0.2546
Optimal (using F1-score) Threshold: 0.1891
F1 Score: 33.33%
Precision: 31.58%  |  Recall: 35.29%
Confusion Matrix:
 [[221  13]
 [ 11   6]]

--- Fold 2 ---
ROC AUC: 0.7793
PR AUC: 0.3636
Optimal (using F1-score) Threshold: 0.1481
F1 Score: 37.21%
Precision: 30.77%  |  Recall: 47.06%
Confusion Matrix:
 [[216  18]
 [  9   8]]

--- Fold 3 ---
ROC AUC: 0.8122
PR AUC: 0.2712
Optimal (using F1-score) Threshold: 0.1540
F1 Score: 37.21%
Precision: 30.77%  |  Recall: 47.06%
Confusion Matrix:
 [[216  18]
 [  9   8]]

--- Fold 4 ---
ROC AUC: 0.7340
PR AUC: 0.1681
Optimal (using F1-score) Threshold: 0.1486
F1 Score: 33.33%
Precision: 25.81%  |  Recall: 47.06%
Confusion Matrix:
 [[211  23]
 [  9   8]]

--- Fold 5 ---
ROC AUC: 0.7011
PR AUC: 0.2084
Optimal (using F1-score) Threshold: 0.1018
F1 Score: 28.17%
Precision: 18.18%  |  Recall: 62.50%
Confusion Matrix:
 [[189  45]
 [  6  10]]

=============== Training Summary ===============

Average ROC AUC (Ra

### **Ε) Επανεκπαίδευση Μοντέλου**

### **ΣΤ) Testing Μοντέλου**

---

## **Μέρος 2ο – Ταξινόμηση σε Υπερφασματική Εικόνα**

### **a.**

### **b.**

### **c.**